<a href="https://colab.research.google.com/github/smaddis01/smad/blob/main/SamuelAddis_Project2Part1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


#Samuel Addis
#CS670
#Project 2 Part 1 Submission

# Mansion Layout:

1. Create a representation of the mansion layout with different rooms (e.g., kitchen, library, ballroom, etc.). This layout can be represented abstractly (e.g., a graph where rooms are nodes and passages are edges) or concretely (e.g., a 2D grid or array). Ensure that the connections between rooms (i.e., which rooms are adjacent and allow movement) are clearly defined. Also, include secret passages between certain rooms as per the original game rules (e.g., Study to Kitchen, Conservatory to Lounge etc.).

2. You may choose between one of two architectures.
    1. Abstract Graph-Based (Recommended): Treat the mansion purely as a graph where rooms are nodes and edges have a "distance cost." A player (Human or AI) can successfully transition along an edge to an adjacent room if their dice roll meets or exceeds that edge's movement cost. in this case you are recommended to use the following weights (path cost) on the mansion graph.
      - Weighted abstract graph configuration
MANSION_GRAPH = {
    "Study": {"Library": 3, "Hall": 3, "Kitchen": 0},          # 0 cost = Secret Passage
    "Library": {"Study": 3, "Billiard Room": 3},
    "Billiard Room": {"Library": 3, "Conservatory": 3},
    "Conservatory": {"Billiard Room": 3, "Ballroom": 4, "Lounge": 0}, # 0 cost = Secret Passage
    "Ballroom": {"Conservatory": 4, "Kitchen": 4},
    "Kitchen": {"Ballroom": 4, "Dining Room": 3, "Study": 0},  # 0 cost = Secret Passage
    "Dining Room": {"Kitchen": 3, "Lounge": 3},
    "Lounge": {"Dining Room": 3, "Hall": 4, "Conservatory": 0}, # 0 cost = Secret Passage
    "Hall": {"Study": 3, "Lounge": 4}
}
2. Character & Weapon Definition:
    - Define the 6 standard Cluedo characters: Miss Scarlett, Colonel Mustard, Mrs. White, Reverend Green, Mrs. Peacock, and Professor Plum. Assign each character a starting position on the board.
    - Define the 6 standard Cluedo weapons: Candlestick, Dagger, Lead Pipe, Revolver, Rope, and Wrench.
      - You can use the following python code:
SUSPECTS = ["Miss Scarlett", "Colonel Mustard", "Mrs. White", "Reverend Green", "Mrs. Peacock", "Professor Plum"]
WEAPONS = ["Candlestick", "Dagger", "Lead Pipe", "Revolver", "Rope", "Wrench"]
ROOMS = list(MANSION_GRAPH.keys())
STARTING_POSITIONS = {
"Miss Scarlett": "Lounge",
"Colonel Mustard": "Dining Room",
"Mrs. White": "Kitchen",
"Reverend Green": "Ballroom",
"Mrs. Peacock": "Conservatory",
"Professor Plum": "Library"
}
3. Solution Selection:
    - From the full set of Character, Weapon, and Room cards, randomly select one of each category to form the "murder solution" (the three cards placed in the confidential envelope). These cards should be hidden from all players throughout the game.
    - After the solution is selected, the remaining cards (the "deck") must be shuffled and distributed evenly among all active players. If the number of cards doesn't divide evenly, some players may receive one more card than others.

# Player Movement
1. Turn Structure:
    - Implement a turn-based system.
    - Turn Order Mandate: Each player takes a turn in sequence. The turn loop must strictly follow a fixed, clockwise sequence based on the index order of the provided SUSPECTS list. Miss Scarlett always takes the first turn of the game. If an intermediate character is not active (for example, in a 3-to-5 player game), the loop must automatically skip that character and proceed to the next active suspect in the sequence, cycling back to the beginning of the list after Professor Plum.
    - A standard Cluedo turn involves:
      - Dice Roll: Simulate rolling a standard 6-sided die for player movement. Your program must simulate rolling a standard 6-sided die. You should use a random number generator (e.g., random.randint(1, 6) in Python) to determine the player's maximum movement capability at the start of every turn. Your program should print the result of the roll clearly to the console (e.g., "Player X rolled a 4!").
      - Movement: Allow the player to move their character token a number of spaces equal to their dice roll. Players can move horizontally or vertically.
      - Entering Rooms: If a player enters a room, their movement for that turn ends. They can then make a suggestion.
      - Exiting Rooms: A player must roll the dice to exit a room on their next turn. They cannot simply move out. Secret passages do not require a dice roll for movement.
      - Validation Rule: Your program should print a clear message if a move is invalid. If a player attempts to move to an adjacent room but rolls a number lower than that edge's distance cost, the program should print an error (e.g., "Roll too low! Move invalid."). The player must then be prompted to either choose an alternative valid path that fits within their roll (such as an available cost-0 secret passage) or forfeit their movement and remain in their current room for that turn.
2. Suggestions:
    - When a player enters a room (either by moving or via a secret passage), they must make a suggestion.
    - A suggestion involves a Character, a Weapon, and the Room the suggesting player just entered. For example, "I suggest it was Colonel Mustard, with the Revolver, in the Study."
    - Character Movement During Suggestion: If the suggested Character is currently on the board and not in the room where the suggestion is being made, their token must be immediately moved into that room. This is a crucial rule for the classic game. The weapon token also moves into that room. For instance, If Player A is in the Lounge and suggests Professor Plum, Professor Plum's token is immediately dragged into the Lounge. and When it becomes Professor Plum’s actual turn next, he does not start from his old room. He must start his turn directly from the Lounge. He can choose to immediately make a suggestion in the Lounge on his turn without rolling the dice, or he can choose to roll the dice to exit.
    - The game should prompt the player for their suggestion.

In [1]:
#Part 1: Mansion Layout and Solution Selection/Hand Dealing
import random

#Mansion Layout
MANSION_GRAPH = {
    "Study": {"Library": 3, "Hall": 3, "Kitchen": 0}, # 0 cost = Secret Passage
    "Library": {"Study": 3, "Billiard Room": 3},
    "Billiard Room": {"Library": 3, "Conservatory": 3},
    "Conservatory": {"Billiard Room": 3, "Ballroom": 4, "Lounge": 0}, # 0 cost = Secret Passage
    "Ballroom": {"Conservatory": 4, "Kitchen": 4},
    "Kitchen": {"Ballroom": 4, "Dining Room": 3, "Study": 0}, # 0 cost = Secret Passage
    "Dining Room": {"Kitchen": 3, "Lounge": 3},
    "Lounge": {"Dining Room": 3, "Hall": 4, "Conservatory": 0}, # 0 cost = Secret Passage
    "Hall": {"Study": 3, "Lounge": 4}
    }

#Character and Weapon Representation
SUSPECTS = ["Miss Scarlett", "Colonel Mustard", "Mrs. White", "Reverend Green", "Mrs. Peacock", "Professor Plum"]
WEAPONS = ["Candlestick", "Dagger", "Lead Pipe", "Revolver", "Rope", "Wrench"]
ROOMS = list(MANSION_GRAPH.keys())
STARTING_POSITIONS = { "Miss Scarlett": "Lounge", "Colonel Mustard": "Dining Room",
                      "Mrs. White": "Kitchen", "Reverend Green": "Ballroom",
                       "Mrs. Peacock": "Conservatory", "Professor Plum": "Library" }

In [5]:
#define players, their attributes
class Player:
  def __init__(self, character, position, hand=None, active=True,
               dragged_in=False, is_human=False):
    self.character = character
    self.position = position
    self.hand = hand if hand is not None else []
    self.active = active
    self.dragged_in = dragged_in
    self.is_human = is_human
  def __repr__(self):
    return f"<{self.character} in {self.position}, hand={self.hand}>"

#creates/defines individual player
def make_player(character, starting_position):
  return Player(character=character, position=starting_position)

#creates/defines/stores all players in the game, assign them starting positions
def make_players(suspects, starting_positions, human_characters):
  PLAYERS = []
  for character in SUSPECTS:
    player = make_player(character, starting_positions[character])
    player.is_human = character in human_characters
    PLAYERS.append(player)
  return PLAYERS

In [10]:
def initialize_game(PLAYERS, SUSPECTS, WEAPONS, ROOMS):
  #initialize game with randomly selected murderer, weapon, and room
  murderer = random.choice(SUSPECTS)
  weapon = random.choice(WEAPONS)
  room = random.choice(ROOMS)
  #define as solution
  solution = {"suspect": murderer, "weapon": weapon, "room": room}

  deck = (
      [s for s in SUSPECTS if s != murderer] #disclude murderer card from deck
      + [w for w in WEAPONS if w != weapon] #disclude murder weapon card from deck
      + [r for r in ROOMS if r != room] #disclude murder room from deck
  )
  random.shuffle(deck)

  #deal hands to all participating players
  while deck:
    for player in PLAYERS:
      if deck:
        card = deck.pop(0)
        player.hand.append(card)

  return solution

#players = make_players(SUSPECTS, STARTING_POSITIONS, human_characters=["Miss Scarlett"])
#solution = initialize_game(players, SUSPECTS, WEAPONS, ROOMS)

In [12]:
#Part 2: Player Movement

#Implement a turn based system.
#Each player makes turn in strict clockwise sequence.
#If intermediate character is inactive, loop must skip that character.

def take_turn(player, MANSION_GRAPH, players):
  print(f"\n{player.character}'s turn (currently in {player.position})")
  #dice roll
  roll = random.randint(1, 6)
  print(f"{player.character} rolled a {roll}")
  #movement
  neighbors = MANSION_GRAPH[player.position]
  destination = random.choice(list(neighbors.keys()))
  cost = neighbors[destination]

  if cost == 0: #validation rule and exiting rooms
    print(f"{player.character} uses a secret passage int {destination}")
    player.position = destination
  elif roll >= cost: #if acceptable, player will move to new destination
    print(f"{player.character} moves into {destination}")
    player.position = destination
  else: #invalidate move if roll too low, barring circumstance of secret passage
    print("Roll too low, move invalid.")
    passages = [r for r, c in neighbors.items() if c == 0]
    if passages:
      print(f"{player.character} uses a secret passage into {passages[0]} instead")
      player.position = passages[0]
    else:
      print(f"{player.character} forfeits movement and stays in {player.position}")
  #if possible, player enters new room, which is denoted as new position assumed
  print(f"{player.character}'s turn ends in {player.position}")

#for player in players:
#    take_turn(player, MANSION_GRAPH, players)

In [13]:
def prompt_suggestion(player):
  print(f"\n{player.character}, you are in {player.position}. Make a suggestion.")
  #randomize suggestion choices for non human players
  if not player.is_human:
    return random.choice(SUSPECTS), random.choice(WEAPONS)
  print(f"\n{player.character}, make a suggestion in the {player.position}")
  print("Suspects:", ", ".join(SUSPECTS))
  print("Weapons:", ", ".join(WEAPONS))

  #provide input field for user, prompt user to suggest suspect and weapon
  while True:
    raw = input("Enter suggestion:").strip() #suggestion box
    parts = [p.strip() for p in raw.split(",")]
    if len(parts) == 2 and parts[0] in SUSPECTS and parts[1] in WEAPONS:
      return parts[0], parts[1] #confirm response is valid
    print("Invalid format.")

#Implement suggestions.
def make_suggestion(player, players):
  room = player.position
  suspect, weapon = prompt_suggestion(player)
  #print suggestion
  suggestion = {"suspect": suspect, "weapon": weapon, "room": room}
  print(f"{player.character} suggests: '{suspect}, with the {weapon}, in {room}.'")

  suggested_player = next(p for p in players if p.character == suspect)
  #move suggested player position to suggested room
  if suggested_player.position != room:
    suggested_player.position = room
    suggested_player.dragged_in = True
    print(f"{suspect} has been moved into {room}.")
  #resolve disproof by suspect
  disprover = None
  shown_card = None

  #incorporate indexing for suspects, show card if suspect is named in order to disprove
  start_idx = SUSPECTS.index(player.character)
  order = SUSPECTS[start_idx+1:] + SUSPECTS[:start_idx+1]

  for name in order:
    candidate = next(p for p in players if p.character == name)
    if candidate is player:
      continue
    matches = [card for card in candidate.hand if card in suggestion.values()]
    if matches:
      disprover = candidate
      shown_card = matches[0]
      break

  if disprover is not None:
    print(f"{disprover.character} shows {player.character} card privately: {shown_card}")
  else:
    print("No one could disprove suggestion")

  return{"suggestion": suggestion, "disprover": disprover, "shown_card": shown_card}

In [14]:
#make accusation
def prompt_accusation(player):
  #allow for cpu players to call accusations at randoom
  if not player.is_human:
    if random.random() < 0.1:
      return random.choice(SUSPECTS), random.choice(WEAPONS), random.choice(ROOMS)
    return None

  #promp player to input if they would like to make accusation
  choice = input(f"\n{player.character}, do you want to make an accusation? (yes/no):").strip().lower()
  if choice not in ("yes", "y"):
    return None

  #provide lists of suspects, weapons, rooms to choose from
  print("Suspects:", ", ".join(SUSPECTS))
  print("Weapons:", ", ".join(WEAPONS))
  print("Rooms:", ", ".join(ROOMS))

  #prompt user to input accusation, verify validity of terms inputted
  while True:
    raw = input("Enter accusation as: Character, Weapon, Room: ").strip()
    parts = [p.strip() for p in raw.split(",")]
    if (len(parts) == 3
        and parts[0] in SUSPECTS
        and parts[1] in WEAPONS
        and parts[2] in ROOMS):
      return parts[0], parts[1], parts[2]
    #throw error if invalid
    print("Invalid input.")

#function for applying accusation to solution
def make_accusation(player, solution):
  guess = prompt_accusation(player)
  if guess is None:
    print(f"{player.character} isn't accusing.")
    return False

  suspect, weapon, room = guess
  print(f"{player.character} makes accusation: '{suspect}, {weapon}, {room}")
  #define correct accusation
  if (suspect, weapon, room) == (solution["suspect"], solution["weapon"], solution["room"]):
    print(f"{player.character} is correct! Puzzle has been solved.")
    return True
  #define incorrect accusation
  else:
    print(f"{player.character} is incorrect and has been eliminated from the game.")
    player.active = False
    return False

In [15]:
#implement user interaction with game
#use text-based CLI

def run_game(players, mansion_graph, solution):
  print("Welcome to Cluedo. The game will begin.")
  game_over = False
  winner = None
  round = 0

  #loop when round ends
  while not game_over:
    round += 1
    print(f"round {round}")

    for player in players:
      if not player.active:
        continue
      take_turn(player, mansion_graph, players) #allow turn for each player
      make_suggestion(player, players) #allow suggestion at each turn
      if make_accusation(player, solution): #end game if correct accusation made
        game_over = True
        winner = player.character
        break

    #game ends if no players remaining
    if not any(p.active for p in players):
      print("all players have been eliminated with incorrect guesses")
      game_over = True
  print("game over")

  if winner:
    print(f"Winner: {winner}")
  else:
    print("no winner")
  return winner

In [16]:
players = make_players(SUSPECTS, STARTING_POSITIONS, human_characters = ["Miss Scarlett"])
solution = initialize_game(players, SUSPECTS, WEAPONS, ROOMS)

me = next(p for p in players if p.character == "Miss Scarlett")
print("Your hand is Miss Scarlett:", me.hand)

run_game(players, MANSION_GRAPH, solution)

Your hand is Miss Scarlett: ['Hall', 'Revolver', 'Lounge']
Welcome to Cluedo. The game will begin.
round 1

Miss Scarlett's turn (currently in Lounge)
Miss Scarlett rolled a 4
Miss Scarlett uses a secret passage int Conservatory
Miss Scarlett's turn ends in Conservatory

Miss Scarlett, you are in Conservatory. Make a suggestion.

Miss Scarlett, make a suggestion in the Conservatory
Suspects: Miss Scarlett, Colonel Mustard, Mrs. White, Reverend Green, Mrs. Peacock, Professor Plum
Weapons: Candlestick, Dagger, Lead Pipe, Revolver, Rope, Wrench


KeyboardInterrupt: Interrupted by user